# Hyperrectangle verification

`NLP-Robustness.ipynb` trained classifiers of the western cluster's choices, a base model and a
PGD-trained model, and exported them to `models/western_base.onnx` and `models/western_pgd.onnx`
(and one base model per other cluster). **Run it first.**

PGD only tries a few points in each hyperrectangle. Here Vehicle and Marabou check **every**
point: a box is verified if every point inside it gets the choice the cluster made for the
dilemma the box was built from. Since each choice is an LTL constraint on who is protected, a
verified box means:

> every embedding in this box leads to the same constraint, e.g. `G ! kill_pedestrians_ahead`.

***
# **VEHICLE SPECIFICATION**

`spec.vcl` states the property for one box. Like the ACAS Xu properties, it names a region of
inputs (the hyperrectangle) and constrains the output there (the box's choice must have the
highest score). The box and the cluster's choice are passed in when Vehicle is called.

In [1]:
%%bash
cat spec.vcl

--------------------------------------------------------------------------------
-- Robustness of the dilemma classifier inside a hyperrectangle

-- The classifier maps the 30 reduced embedding dimensions of a Moral Machine
-- dilemma's description to scores for the two choices, stay or swerve. A
-- hyperrectangle encloses the embeddings of a dilemma and its character
-- perturbations. The network is robust on the box if every point inside it gets
-- the choice the cultural cluster made for that dilemma.

--------------------------------------------------------------------------------
-- Inputs

inputSize = 30
type Input = Tensor Real [inputSize]

-- The embeddings were scaled to [0, 1] in every dimension.

validInput : Input -> Bool
validInput x = forall i . 0 <= x ! i <= 1

--------------------------------------------------------------------------------
-- Outputs

-- One score per choice; the choice with the highest score is made.

type Output = Tensor Real [2]
type Label = Index 2


***
# **VERIFICATION**

`verify.sh` verifies one box: `./verify.sh [box index] [network.onnx] [cluster]`. It writes the
box to a temporary file in the IDX format Vehicle reads datasets from, looks up the cluster's
choice for its dilemma, and calls `vehicle verify`.

In [2]:
%%bash
cat verify.sh

#!/bin/bash
# Verify the classifier on one hyperrectangle with Vehicle + Marabou (spec.vcl).
#
#   ./verify.sh [box index] [network.onnx] [cluster]
#
# Defaults: box 0, models/western_base.onnx and the western cluster's choices. The box and
# the cluster's choice for its dilemma (0 stay, 1 swerve) are read from
# embeddings/hyperrectangles.npz. To verify many boxes at once, use src/verify_boxes.py.

set -e
cd "$(dirname "$0")"

# Use vehicle and Marabou from this folder's .venv, whether or not it is activated
if [ -d .venv/bin ]; then
  PATH="$(cd .venv/bin && pwd):$PATH"
fi

BOX=${1:-0}
NETWORK=${2:-models/western_base.onnx}
CLUSTER=${3:-western}

# Write the box to a temporary IDX file, the format Vehicle reads datasets from
BOX_FILE=$(mktemp --suffix=.idx)
trap 'rm -f "$BOX_FILE"' EXIT
LABEL=$(python src/verify_boxes.py --export-box "$BOX" "$BOX_FILE" --cluster "$CLUSTER")
echo "Box $BOX, $CLUSTER choice $LABEL (0 stay, 1 swerve)"
if [ "$LABEL" -lt 0 ]; then
  echo "The $CLUSTER clu

In [3]:
%%bash
# Verify box 271 on the base model
./verify.sh 271 models/western_base.onnx western

Box 271, western choice 0 (0 stay, 1 swerve)
  robust [=========================================================] 1/1 queries
Verifying properties:
    result: ✗ - Marabou found a counterexample
      x: [ 0.457149, 0.292144, 0.418225, 0.155108, 0.326885, 0.637077, 0.352659, 0.725438, 0.603517, 0.632201, 0.546496, 0.379655, 0.628053, 0.534486, 0.212138, 0.299222, 0.525757, 0.659514, 0.325889, 0.264937, 0.563996, 0.342554, 0.712054, 0.628562, 0.655426, 0.719475, 0.624335, 0.624151, 0.736338, 0.510399 ]


Box 271 was built from this dilemma and its five character typos:

> A self-driving car has sudden brake failure. If it stays on course, it will drive through the pedestrian crossing ahead, killing one boy and one male executive. If it swerves, it will drive through the pedestrian crossing in the other lane, killing one girl and one female executive.

Most western respondents chose to **stay** (choice 0).

**On the base model, box 271 fails.** Marabou returns a counterexample: 30 numbers, one per
input dimension, for a point inside the box that the base model gives the other choice.

A counterexample is a point in **embedding space**, not a sentence. It lies between the embeddings
of the dilemma's typos, but no sentence need embed to exactly that point. This is the gap between
embedding space and sentences that the NLP verification literature discusses.

In [4]:
%%bash
# Verify box 271 on the PGD-trained model
./verify.sh 271 models/western_pgd.onnx western

Box 271, western choice 0 (0 stay, 1 swerve)
  robust [=========================================================] 1/1 queries
Verifying properties:
    result: 🗸 - Marabou proved no counterexample exists


**On the PGD-trained model, box 271 is verified.** Marabou proves that every point inside the
box, not just the ones PGD tried, gets the western choice. Adversarial training pushed the decision
boundary out of this box.

***
# **VERIFYING MANY BOXES**

Vehicle 0.28 cannot quantify over all boxes in one property (see the comment in `spec.vcl`), so
`src/verify_boxes.py` calls `vehicle verify` once per box and counts the results. Most boxes take
under a second, but Marabou needs much longer for a few, so each box gets at most 60 seconds
(`--timeout`) and otherwise counts as a timeout. The cells below check the same random sample of
100 boxes for each model. Remove `--sample 100` to check all 1,834 (20 to 30 minutes per model).

In [5]:
%run src/verify_boxes.py models/western_base.onnx --cluster western --sample 100

Boxes checked: 100
Verified (no counterexample): 2
Counterexample found: 98
Timeout (no answer within 60s): 0
Errors: 0


In [6]:
%run src/verify_boxes.py models/western_pgd.onnx --cluster western --sample 100

Boxes checked: 100
Verified (no counterexample): 17
Counterexample found: 82
Timeout (no answer within 60s): 1
Errors: 0


Over all 1834 boxes:

| | base | PGD-trained |
|---|---|---|
| verified | 48 | 220 |
| of which 'stay' / 'swerve' | 45 / 3 | 220 / 0 |
| counterexample | 1784 | 1596 |
| timeout (60 s) | 2 | 18 |

**PGD training verifies over four times as many boxes, but every one of them is a 'stay' box.**
The western cluster chose 'stay' on 57% of dilemmas, yet the PGD-trained model verifies no
'swerve' box at all: its robustness comes entirely from leaning towards 'stay', as
`NLP-Robustness.ipynb` showed. A verified box is only as meaningful as the boxes it is built from.

Two things limit what these results mean:

* **The boxes contradict each other.** 43% of the boxes intersect a box with the opposite choice,
  so no network can verify all of them. Character typos are a poor way to build boxes for text
  that always uses the same template.
* **A verified box is not a verified sentence.** Most points in a box are not the embedding of
  any real dilemma description.

***
# **COURSEWORK**

The worked example follows ANTONIO: boxes from **character typos** in the words that name the
characters, verified against the **western** cluster's choices. The coursework extends it.

1. **Build better boxes.** Character typos give boxes that contradict each other. Try:
   * **Synonyms**: describe the same characters in other words ("elderly" -> "old", "large" ->
     "overweight", "female athlete" -> "sportswoman", "baby in a stroller" -> "infant in a pram").
     The meaning is unchanged by construction.
   * **Epsilon-cubes**: ANTONIO's other kind of box, a cube of side 2ε around each embedding.
     Try several values of ε.

   For each kind of box, measure how many boxes contradict each other (`conflicting_boxes` in
   `src/hyperrectangles.py`), how many survive PGD, and how many verify, with and without PGD
   training. What does a verified epsilon-cube say about sentences? New perturbations need
   `requirements-embed.txt` and a change to `src/prepare.py`.

2. **Cultural alignment.** Verify the eastern and southern models against their own cluster's
   choices (`--cluster eastern`, `--cluster southern`). Focus on the dilemmas where the clusters
   disagree: is a cluster's model *robustly* aligned with its own cluster there, or only on the
   exact sentence? Box 298, for example, was built from a dilemma where the western and eastern
   clusters stay and the southern cluster swerves. Can you train a model that verifies on one
   cluster's choices but not another's?
   This is the question of the position paper *Exploring Verification Frameworks for Social Choice
   Alignment* (Ciupa, Belle and Komendantskaya, NeSy 2025).

3. **Word perturbations.** ANTONIO also perturbs whole words: deletion, repetition, negation,
   reordering, singular/plural and verb tense. Which ones keep the dilemma the same? Swapping
   "ahead" and "in the other lane", or changing a number, changes the dilemma and should change the
   choice. Build word hyperrectangles and compare.

4. **Explain the classifier with LIME.** Use LIME's text explainer on the whole pipeline
   (sentence -> `src/embed.py` -> network), so that importance is given per word. Does the network
   rely on the characters the Moral Machine found people care about (humans over pets, more lives
   over fewer, the young over the old), or on something else? This needs `requirements-embed.txt`.

***
# **EXTENSION: CONDITIONAL AND TEMPORAL SPECIFICATIONS**

Every label so far is a one-step safety formula, `G ! kill_X`: a Moral Machine dilemma is a single
decision, with nothing before or after it. Ethical Reward Machines (Ciupa and Belle, NeSy 2024)
work with richer, temporally extended specifications. This open-ended extension moves towards
them.

**(a) Conditional rules across dilemmas.** Each dilemma records its conditions: who is ahead,
who is in the other lane (children, elderly people, pets, passengers), and whether the pedestrians
are crossing on red. Look for rules a cluster follows across many dilemmas, and state them as
conditional LTL formulas, for example:

```
G ( (pets_ahead & humans_in_other_lane) -> stay_is_forbidden )    spare humans over pets
G ( (pedestrians_ahead & crossing_on_red) -> ! swerve )            do not swerve to save people crossing on red
G ( children_ahead -> ! kill_children )                            spare the young
```

These are Awad et al.'s factors (species, legality, age, ...) written as norms. Use the vote counts
in `dataset/votes.csv.gz` to measure how consistently each cluster follows each rule. The same
formula may be a norm for one culture and not for another. Then train a classifier of which rule
applies to a dilemma, and verify it as above. Does a cluster's model follow the rule robustly, or
only dilemma by dilemma?

**(b) Temporally extended specifications.** Temporal operators such as `F` (eventually) and `U`
(until) need a sequence of events, which the Moral Machine does not provide. Design sequences
around the dilemma, for example its aftermath:

```
G ( collision -> F stop )
G ( kill_pedestrians -> F call_emergency_services )
! leave_scene U police_arrived
```

Translate the formulas into a reward machine: an automaton, built for example with Spot or an
LTLf-to-DFA tool, whose states give an RL agent temporally extended rewards in a grid world. A
cluster's formulas then give a culture-specific reward machine.

Be careful about what this shows:

* The sequences are your design, not data: justify them.
* Verifying the classifier certifies that it picks the same formula everywhere in a box. It does not
  prove that an agent satisfies the formula over time; the reward machine's automaton, or a model
  checker, is the tool for that.
* More formulas mean more classes. Classifying the formula's shape and filling in its propositions
  separately keeps the network small enough to verify.